# Generate PDBs from NPB Epoch 100 Stacked Reconstructions

Runs the structure module on `sae_reconstructions_epoch_100/{id}_reconstructed_pair_block_47.npy` (stacked NPB epoch 100).

Output: `npb_epoch_100_pdbs/{id}_reconstructed_pair_block_47_structure.pdb`

In [ ]:
# Imports and helpers
import os
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np

LAYER = 47
PROJECT_ROOT = Path(".").resolve()


def find_fasta(subdir: Path, protein_id: str) -> Optional[Path]:
    for fn in [f"{protein_id}.fasta", "seq.fasta"]:
        p = subdir / fn
        if p.is_file():
            return p
    for f in subdir.iterdir():
        if f.suffix == ".fasta":
            return f
    return None

print("✅ Imports OK")

In [ ]:
# Configuration
BASE = os.environ.get("BASE", "/storage/scratch1/5/sshrestha304/Autoencoder/CompleteProteins")
RECON_DIR = os.path.join(BASE, "sae_reconstructions_epoch_100")
OUTPUT_DIR = os.path.join(BASE, "npb_epoch_100_pdbs")
MODEL_DEVICE = "cpu"  # or "cuda", "cuda:0"
LIMIT = None  # None = all; set to e.g. 3 for testing
DRY_RUN = False

base = Path(BASE).resolve()
recon_dir = Path(RECON_DIR).resolve()
output_dir = Path(OUTPUT_DIR).resolve()
run_script = PROJECT_ROOT / "run_structure_module.py"

if not run_script.is_file():
    raise FileNotFoundError(f"run_structure_module.py not found at {run_script}")
if not recon_dir.is_dir():
    raise FileNotFoundError(f"Recon dir not found: {recon_dir}")

print(f"Base:      {base}")
print(f"Recon dir: {recon_dir}")
print(f"Output:    {output_dir}")
print(f"Device:    {MODEL_DEVICE}")

In [ ]:
# Discover stacked reconstructions
suffix = "_reconstructed_pair_block_47.npy"
pairs = []
for f in sorted(recon_dir.iterdir()):
    if f.name.endswith(suffix):
        protein_id = f.name.replace(suffix, "")
        pairs.append((protein_id, f))

if not pairs:
    raise SystemExit(f"No {suffix} found in {recon_dir}")

total = len(pairs)
if LIMIT:
    pairs = pairs[:LIMIT]
    print(f"Processing first {len(pairs)} of {total}")
else:
    print(f"Found {total} proteins")

output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Process each protein
subdir = base  # protein folders are under base, e.g. base/6tf4_A/
ok = 0
fail = 0

for i, (protein_id, pair_npy) in enumerate(pairs, 1):
    protein_subdir = base / protein_id
    single_npy = protein_subdir / f"single_block_{LAYER}.npy"
    aatype_npy = protein_subdir / f"aatype_{LAYER}.npy"
    fasta_path = find_fasta(protein_subdir, protein_id)

    cmd = [
        sys.executable,
        str(run_script),
        str(pair_npy),
        str(output_dir),
        "--model-device",
        MODEL_DEVICE,
    ]
    if single_npy.is_file():
        cmd.extend(["--single-npy", str(single_npy)])
    else:
        cmd.append("--single-from-pair")

    temp_aatype = None
    if fasta_path:
        cmd.extend(["--fasta", str(fasta_path)])
    elif aatype_npy.is_file():
        cmd.extend(["--aatype-npy", str(aatype_npy)])
    else:
        pair_arr = np.load(pair_npy, allow_pickle=True)
        n_res = pair_arr.shape[0] if pair_arr.ndim >= 2 else pair_arr.shape[1]
        fd, temp_aatype = tempfile.mkstemp(suffix=".npy")
        os.close(fd)
        np.save(temp_aatype, np.zeros(n_res, dtype=np.int64))
        cmd.extend(["--aatype-npy", temp_aatype])

    try:
        if DRY_RUN:
            print(f"[{i}/{len(pairs)}] Would run: {protein_id}")
            ok += 1
        else:
            print(f"[{i}/{len(pairs)}] {protein_id}...", end=" ", flush=True)
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            out_name = f"{pair_npy.stem}_structure.pdb"
            out_pdb = output_dir / out_name
            if result.returncode == 0 and out_pdb.is_file():
                print(f"✅ {out_name}")
                ok += 1
            else:
                err = result.stderr[:200] if result.stderr else result.stdout[:200]
                print(f"❌ {err}")
                fail += 1
    except subprocess.TimeoutExpired:
        print("❌ Timeout")
        fail += 1
    except Exception as e:
        print(f"❌ {e}")
        fail += 1
    finally:
        if temp_aatype and os.path.exists(temp_aatype):
            try:
                os.unlink(temp_aatype)
            except OSError:
                pass

print(f"\nDone: {ok} OK, {fail} failed")